In [4]:

# Practical 4D: Recommendation System using CBOW

# 1. Import libraries
import pandas as pd
import numpy as np
import re
from gensim.models import Word2Vec

# 2. Load dataset
data = pd.read_csv("twitter_dataset.csv")

print("Available Columns:", data.columns.tolist())

# 3. Automatically select tweet column
possible_columns = ["text", "tweet", "tweets", "content", "message"]

text_column = next(
    (col for col in data.columns if col.strip().lower() in possible_columns),
    data.columns[-1]
)

tweets = data[text_column].dropna().astype(str)

print("Selected Column:", text_column)

# 4. Clean and tokenize tweets
stop_words = {"the", "is", "a", "an", "and", "to", "of",
              "in", "it", "this", "that", "for", "on",
              "with", "are", "was", "be"}

sentences = []

for tweet in tweets:
    tweet = re.sub(r"[^a-z\s]", "", tweet.lower())
    words = [w for w in tweet.split() if w not in stop_words]

    if words:
        sentences.append(words)

if not sentences:
    raise ValueError("No valid tweets found!")

print("Total Tweets:", len(sentences))

# 5. Train CBOW model
cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=0,
    epochs=10
)

print("CBOW Model Trained Successfully!")

# 6. Convert tweets into average word vectors
def get_vector(words):
    vectors = [cbow_model.wv[w] for w in words if w in cbow_model.wv]

    if not vectors:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

tweet_vectors = np.array([get_vector(words) for words in sentences])

# 7. Take user input
user_input = input("Enter your search keywords: ").lower()
user_input = re.sub(r"[^a-z\s]", "", user_input)

input_words = [w for w in user_input.split() if w not in stop_words]
input_vector = get_vector(input_words)

# 8. Calculate cosine similarity
similarities = []

for vector in tweet_vectors:
    similarity = np.dot(input_vector, vector) / (
        np.linalg.norm(input_vector) * np.linalg.norm(vector) + 1e-10
    )
    similarities.append(similarity)

# 9. Display top 5 recommendations
top_indices = np.argsort(similarities)[::-1][:5]

print("\nRecommended Tweets:")

for i in top_indices:
    print("\nTweet:", " ".join(sentences[i]))
    print("Similarity:", round(similarities[i], 4))


Available Columns: ['Tweet_ID', 'Username', 'Text', 'Retweets', 'Likes', 'Timestamp']
Selected Column: Text
Total Tweets: 10000
CBOW Model Trained Successfully!

Recommended Tweets:

Tweet: body onto understand team about product beautiful might class picture guy significant resource material minute begin happen adult old carry pay government agree less night family maybe side process strategy choose until push respond agree close style as
Similarity: 0.0

Tweet: guess without successful save particular natural management increase every large production decade ready once must likely run piece act view young certainly look enjoy stage create soon natural statement laugh material many reason artist never
Similarity: 0.0

Tweet: you day agent likely region teacher data message build as third child forward her recent admit order black trip source four into low table data him sing plan assume land talk brother late less air effect reduce line
Similarity: 0.0

Tweet: born which push still de

# Practical 4D – Building a Recommendation System with CBOW Word Embeddings

Problem Statement: Use CBOW word embeddings to build a content-based recommendation system. For example, recommend similar articles or tweets based on the embeddings of keywords in a user's input.

## 1. First understand the logic

The main aim of this practical is to recommend tweets that are similar to what the user searches for.

For example, suppose your Twitter dataset contains these tweets:

1. "I love cricket and watching IPL matches."
2. "Python is a popular programming language."
3. "Virat Kohli is a great cricket player."
4. "Machine learning is used in artificial intelligence."

Now suppose the user enters:

`cricket IPL`

The system should recommend tweets related to cricket and IPL.

How does it work?

1. Load tweets from the CSV file.
2. Clean the tweets and split them into words.
3. Train the CBOW model to learn word embeddings.
4. Convert every tweet into a numerical vector by averaging its word embeddings.
5. Convert the user's search input into a vector in the same way.
6. Calculate cosine similarity between the user's vector and every tweet vector.
7. Display the five most similar tweets.

### Overall flow

Twitter Dataset (CSV)

Clean and tokenize tweets

Train CBOW Word2Vec

Convert tweets into vectors

Take user search input and convert it into a vector

Calculate cosine similarity

Recommend top 5 similar tweets

## 2. Understand the code step by step

### Step 1: Import libraries

```
import pandas as pdimport numpy as npimport refrom gensim.models import Word2Vec
```

- `pandas`: Reads the Twitter CSV dataset.
- `numpy`: Performs numerical calculations and averages vectors.
- `re`: Removes special characters.
- `Word2Vec`: Trains the CBOW model.

### Step 2: Load the dataset

```
data = pd.read_csv("twitter_dataset.csv")print("Available Columns:", data.columns.tolist())
```

- Reads the Twitter dataset.
- Displays the available column names.

For example, your CSV might contain:

| tweet_id | text            | username |
| -------- | --------------- | -------- |
| 1        | I love cricket  | user1    |
| 2        | Learning Python | user2    |

### Step 3: Automatically select the tweet column

```
possible_columns = ["text", "tweet", "tweets", "content", "message"]text_column = next(    (col for col in data.columns if col.strip().lower() in possible_columns),    data.columns[-1])tweets = data[text_column].dropna().astype(str)
```

What is happening?

- Checks whether a column is named `text`, `tweet`, `content`, etc.
- Selects the matching column.
- If no matching column exists, selects the last column.
- `dropna()` removes empty rows.
- `astype(str)` converts values into strings.

Why do we do this?

Because different Kaggle datasets may use different column names for tweets.

### Step 4: Clean and tokenize tweets

```
stop_words = {"the", "is", "a", "an", "and", "to", "of",              "in", "it", "this", "that", "for", "on",              "with", "are", "was", "be"}sentences = []for tweet in tweets:    tweet = re.sub(r"[^a-z\s]", "", tweet.lower())    words = [w for w in tweet.split() if w not in stop_words]    if words:        sentences.append(words)
```

This step prepares tweets for CBOW training.

First, stop words:

Stop words are common words that often carry little topic-specific information.

Examples: `the`, `is`, `and`, `a`, `of`.

Then, cleaning:

```
tweet.lower()
```

Converts all text into lowercase.

```
re.sub(r"[^a-z\s]", "", tweet.lower())
```

Removes numbers, punctuation and special characters.

Then, tokenization:

```
tweet.split()
```

Splits the tweet into individual words.

Finally:

```
sentences.append(words)
```

Stores the cleaned words of each tweet.

Example:

Original tweet:

`"I LOVE Cricket!!! This is amazing."`

After cleaning and removing stop words:

```
['love', 'cricket', 'amazing']
```

### Step 5: Train CBOW model

```
cbow_model = Word2Vec(    sentences=sentences,    vector_size=100,    window=5,    min_count=1,    sg=0,    epochs=10)
```

This is where the model learns relationships between words.

| Parameter         | Meaning                                          |
| ----------------- | ------------------------------------------------ |
| `sentences`       | Cleaned tweets                                   |
| `vector_size=100` | Each word gets 100 numerical values              |
| `window=5`        | Considers surrounding words within a window of 5 |
| `min_count=1`     | Keeps words appearing at least once              |
| `sg=0`            | Uses CBOW                                        |
| `epochs=10`       | Trains the model for 10 epochs                   |

Remember:

- CBOW: Context words predict the target word.
- Skip-gram: Target word predicts surrounding words.

### Step 6: Convert tweets into average word vectors

```
def get_vector(words):    vectors = [cbow_model.wv[w] for w in words if w in cbow_model.wv]    if not vectors:        return np.zeros(100)    return np.mean(vectors, axis=0)
```

Most important step

This function converts a complete tweet into a single numerical vector.

Remember, Word2Vec creates a separate vector for every word. But we need one vector to represent the entire tweet.

So, we calculate the average of all word vectors.

Example:

Suppose a tweet has three words:

- Cricket → vector of 100 numbers
- IPL → vector of 100 numbers
- Match → vector of 100 numbers

We calculate the average of these three vectors to get one 100-dimensional tweet vector.

What each line does:

- `cbow_model.wv[w]`: Gets the embedding of a word.
- `if w in cbow_model.wv`: Checks whether the word exists in the vocabulary.
- `np.zeros(100)`: Returns a zero vector if no words are found.
- `np.mean(vectors, axis=0)`: Calculates the average of the word vectors.

Then:

```
tweet_vectors = np.array([get_vector(words) for words in sentences])
```

Converts every tweet into its average vector and stores all vectors in a NumPy array.

For example, if there are 1,000 tweets:

`tweet_vectors.shape = (1000, 100)`

That means 1,000 tweets, each represented by 100 numbers.

### Step 7: Take user input

```
user_input = input("Enter your search keywords: ").lower()user_input = re.sub(r"[^a-z\s]", "", user_input)input_words = [w for w in user_input.split() if w not in stop_words]input_vector = get_vector(input_words)
```

Now the system asks the user to enter search keywords.

Example:

`Enter your search keywords: cricket IPL`

What happens?

1. Converts the input into lowercase.
2. Removes special characters.
3. Splits the input into individual words.
4. Removes stop words.
5. Uses `get_vector()` to convert the search words into one average vector.

Important: The user's input and tweets are converted using the same CBOW model. This allows us to compare their vectors.

### Step 8: Calculate cosine similarity

```
similarities = []for vector in tweet_vectors:    similarity = np.dot(input_vector, vector) / (        np.linalg.norm(input_vector) * np.linalg.norm(vector) + 1e-10    )    similarities.append(similarity)
```

This is the main recommendation logic.

What is cosine similarity?

Cosine similarity measures how similar two vectors are by calculating the angle between them.

In simple words, it checks how closely the user's search vector matches each tweet vector.

Formula:

\\[ \text{Cosine Similarity} = \frac{A \cdot B}{\\|A\\|\times\\|B\\|} \\]

Where:

- A = User input vector.
- B = Tweet vector.
- \\(A \cdot B\\) = Dot product.
- \\(\\|A\\|\\) and \\(\\|B\\|\\) = Vector magnitudes.

[X](https://images.openai.com/static-rsc-4/LRZROMolyR79Om5txHScfM0djJuSeI0HZ1lRRE5pVOrK-NoZroyQZNs6YAfUVMERCR7spklOqtqcxeHbvlY6SlhF9V13y9OQhGgvnUL61NBd_2GbY00gR9d7v7bvx8N56I-C2kUOIz71_Tl8J4ohF4roxbitW0Qss00lDLuaDHI?purpose=inline)

Understanding the code:

- `np.dot()` calculates the dot product.
- `np.linalg.norm()` calculates the magnitude of a vector.
- `1e-10` is a very small number added to avoid division by zero.
- `similarities.append()` stores the similarity score of each tweet.

A higher cosine similarity generally means the tweet is more similar to the user's input.

### Step 9: Display top 5 recommendations

```
top_indices = np.argsort(similarities)[::-1][:5]
```

This line selects the five tweets with the highest similarity scores.

Understand it:

- `np.argsort(similarities)` sorts the indices according to similarity in ascending order.
- `[::-1]` reverses the order, giving descending order.
- `[:5]` selects the first five indices.

Then:

```
print("\nRecommended Tweets:")for i in top_indices:    print("\nTweet:", " ".join(sentences[i]))    print("Similarity:", round(similarities[i], 4))
```

- Loops through the top five tweets.
- `sentences[i]` gets the recommended tweet's words.
- `" ".join()` combines the words into a sentence.
- `round(..., 4)` displays the similarity score up to four decimal places.

Example output (illustrative):

```
Enter your search keywords: cricket IPL

Recommended Tweets:

Tweet: love cricket watching ipl matches
Similarity: 0.8721

Tweet: virat kohli great cricket player
Similarity: 0.7954
```

The actual recommendations depend on your dataset and trained model.

## 3. Understand the complete recommendation logic

Let's take a simple example.

User searches for: `cricket IPL`

The model converts this input into a 100-dimensional vector.

It has already converted every tweet into a 100-dimensional vector.

Now it compares the user's vector with every tweet vector using cosine similarity.

| Tweet               | Similarity (example) |
| ------------------- | -------------------- |
| Cricket IPL matches | 0.91                 |
| Virat Kohli cricket | 0.82                 |
| Football World Cup  | 0.30                 |
| Python programming  | 0.12                 |

The system sorts the scores and recommends the tweets with the highest similarity.

This is called a content-based recommendation system because recommendations are based on the content (words and meaning represented by embeddings), not on other users' ratings or behaviour.

## 4. Important viva theory questions

Q1. What is a recommendation system?

A recommendation system suggests items to users based on their interests, preferences or input.

Q2. What is a content-based recommendation system?

It recommends items based on the similarity of their content, such as text, descriptions or keywords.

Q3. What is CBOW?

CBOW stands for Continuous Bag of Words. It predicts a target word using surrounding context words.

Q4. What is Word2Vec?

Word2Vec is a technique that converts words into numerical vectors called word embeddings.

Q5. Why do we use CBOW in this practical?

We use CBOW to learn word embeddings from tweets so that we can represent and compare their textual content.

Q6. Why do we calculate the average of word embeddings?

Each word has its own vector. We calculate their average to represent the complete tweet using one fixed-size vector.

Q7. What is cosine similarity?

Cosine similarity measures the similarity between two vectors by calculating the cosine of the angle between them.

Q8. Why do we use cosine similarity instead of Euclidean distance?

Cosine similarity focuses on the direction of vectors rather than their magnitude, making it useful for comparing text embeddings.

Q9. What is the difference between content-based and collaborative filtering?

- Content-based: Recommends items based on their content.
- Collaborative filtering: Recommends items based on the preferences and behaviour of similar users.

Q10. What is the purpose of `np.argsort()`?

It sorts the similarity scores and helps us select the most similar tweets.

Q11. Why do we use `min_count=1`?

It keeps words even if they appear only once in the dataset.

Q12. What is the purpose of stop-word removal?

It removes common words that usually provide less useful information for identifying the topic.

Q13. What does `vector_size=100` mean?

Every word is represented using 100 numerical values.

Q14. What is the purpose of `most_similar()`?

It finds words with similar embeddings. In this practical, we instead use cosine similarity to find similar tweets.

## 5. What to explain to the examiner

You can speak this:

"I implemented a content-based recommendation system using CBOW Word2Vec. First, I loaded the Twitter dataset and selected the tweet column. Then I cleaned the tweets, converted them into lowercase, removed special characters and stop words, and tokenized them.

Next, I trained a CBOW model to generate word embeddings. I converted each tweet into a single vector by taking the average of its word embeddings.

After that, I accepted keywords from the user and converted them into a vector using the same model. I calculated cosine similarity between the user input vector and all tweet vectors. Finally, I sorted the similarity scores and recommended the top five most similar tweets."

## 6. Quick revision before practical

| Concept            | Remember                          |
| ------------------ | --------------------------------- |
| CBOW               | Context words predict target word |
| Word2Vec           | Converts words into vectors       |
| Content-based      | Recommends based on content       |
| Average embedding  | One vector for one tweet          |
| Cosine similarity  | Measures vector similarity        |
| `np.dot()`         | Dot product                       |
| `np.linalg.norm()` | Vector magnitude                  |
| `np.argsort()`     | Sorts indices                     |
| `[::-1]`           | Reverses order                    |
| `[:5]`             | Selects top five                  |

One important code limitation

Your code is suitable for the basic practical, but remember:

- If the user enters words that are not present in the trained vocabulary, `get_vector()` returns a zero vector. The recommendations will then be meaningless.
- If a tweet contains no valid words after cleaning, it can also get a zero vector.
- For a real recommendation system, these cases should be filtered out.

Most important thing to remember: CBOW learns word vectors, average pooling converts each tweet into one vector, and cosine similarity finds the tweets closest to the user's search.